# Semana 1 · ¿Por qué el Sol no sale siempre por el mismo sitio?

Este cuaderno acompaña a la sesión de proyecto de la semana 1. Vas a usar Python como calculadora científica, a construir una maqueta a escala del Sistema Solar y a comprobar con un modelo matemático lo que mides en Stellarium.

**Cómo se usa.** Cada recuadro gris es una celda de código. Selecciónala y pulsa `Mayús + Intro` para ejecutarla; el resultado aparece debajo. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar: no se rompe nada.

Tiempo estimado: 30-45 minutos.

## 1. Python como calculadora científica

Python escribe los números muy grandes o muy pequeños en notación científica con la letra `e`: `1.496e8` significa 1,496 × 10⁸. Los datos son los de la página de la semana (ficha de la Tierra de la NASA y valores nominales de la IAU).

In [ ]:
# Una línea que empieza por # es un comentario: Python la ignora.
ua_km = 1.496e8             # 1 unidad astronómica (distancia media Tierra-Sol), en km
radio_tierra_km = 6371      # radio medio de la Tierra, en km
radio_sol_km = 6.957e5      # radio nominal del Sol, en km
distancia_luna_km = 3.844e5 # distancia media Tierra-Luna, en km

print(f"1 UA en metros: {ua_km * 1000:.3e} m")
print(f"Radio del Sol / radio de la Tierra: {radio_sol_km / radio_tierra_km:.1f}")

`f"..."` es una cadena con huecos: lo que va entre llaves se calcula y se inserta, y `:.3e` pide tres decimales en notación científica, `:.1f` un decimal normal.

**Ejercicio.** Calcula cuántos diámetros de la Tierra caben entre la Tierra y la Luna. Escribe tu cálculo en la celda de abajo antes de mirar la solución.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
diametros = distancia_luna_km / (2 * radio_tierra_km)
print(f"Entre la Tierra y la Luna caben unos {diametros:.1f} diámetros terrestres")

## 2. Una maqueta a escala del Sistema Solar

Una escala es una proporción fija: todo lo real se multiplica por el mismo factor. Si el Sol de la maqueta mide 10 cm de diámetro, el factor es *10 cm / diámetro real del Sol*. Con ese único factor sale todo lo demás.

In [ ]:
diametro_sol_modelo_cm = 10   # cámbialo y vuelve a ejecutar

# factor de escala: cm de maqueta por cada km real
factor = diametro_sol_modelo_cm / (2 * radio_sol_km)

tierra_mm = 2 * radio_tierra_km * factor * 10
distancia_tierra_sol_m = ua_km * factor / 100
distancia_luna_cm = distancia_luna_km * factor

print(f"Sol:  esfera de {diametro_sol_modelo_cm} cm")
print(f"Tierra: esfera de {tierra_mm:.2f} mm, a {distancia_tierra_sol_m:.1f} m del Sol")
print(f"Luna a {distancia_luna_cm:.1f} cm de la Tierra")

La conclusión es incómoda y por eso conviene calcularla: en esta maqueta la Tierra es una cuenta de menos de 1 mm situada a casi 11 metros del Sol. Casi todo el Sistema Solar es espacio vacío, y los dibujos de los libros, que ponen planetas grandes y cercanos, no están a escala.

## 3. Modelo: cuántas horas de luz tiene un día

Ahora el problema de la semana. El modelo necesita dos magnitudes:

- La **declinación del Sol** δ: el ángulo, hacia el norte (positivo) o hacia el sur (negativo), al que se encuentra el Sol respecto al ecuador celeste. Vale +23,44° en el solsticio de junio, −23,44° en el de diciembre y 0° en los equinoccios. El valor 23,44° es la inclinación del eje de la Tierra.
- La **latitud** φ del lugar.

> **Aviso importante.** Las fórmulas de esta sección usan trigonometría (seno, coseno, tangente), que todavía no has estudiado. Por ahora trátalas como una caja negra: no hace falta entender *por qué* funcionan, solo comprobar que *sí* funcionan comparándolas con Stellarium. No las demuestro todavía: la trigonometría llega en las semanas 6 y 7, y las coordenadas del cielo en las semanas 3 y 19, y entonces podrás deducirlas. Son aproximaciones: no incluyen la refracción de la atmósfera ni el tamaño del disco solar, así que las horas de luz reales son unos minutos mayores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

OBLICUIDAD = 23.44   # grados: inclinación del eje de la Tierra respecto a su órbita

def declinacion(dia):
    """Declinación del Sol en grados. dia: día del año (1 de enero = 1). Aproximación (error < 1°)."""
    return -OBLICUIDAD * np.cos(np.radians(360 / 365 * (dia + 10)))

def horas_de_luz(latitud, dia):
    """Horas entre la salida y la puesta del Sol, sin refracción atmosférica."""
    lat = np.radians(latitud)
    dec = np.radians(declinacion(dia))
    cos_h = np.clip(-np.tan(lat) * np.tan(dec), -1, 1)  # fuera de [-1, 1]: 0 h o 24 h
    return 2 * np.degrees(np.arccos(cos_h)) / 15

print("Declinación el 21 de junio (día 172):", round(float(declinacion(172)), 2), "°")
print("Declinación el 21 de diciembre (día 355):", round(float(declinacion(355)), 2), "°")

In [ ]:
dias = np.arange(1, 366)
inicio_mes = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
meses = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]

mi_latitud = 40   # grados, positiva en el hemisferio norte. Cámbiala por la de tu ciudad.

fig, ax = plt.subplots(figsize=(9, 4.5))
for lat in [0, 20, 40, 60]:
    ax.plot(dias, horas_de_luz(lat, dias), label=f"latitud {lat}°")
if mi_latitud not in [0, 20, 40, 60]:
    ax.plot(dias, horas_de_luz(mi_latitud, dias), "k--", label=f"mi latitud ({mi_latitud}°)")
ax.set_xticks(inicio_mes)
ax.set_xticklabels(meses)
ax.set_ylabel("horas de luz")
ax.set_title("Horas de luz a lo largo del año según la latitud (hemisferio norte)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

**Qué mirar.** En el ecuador (latitud 0°) la curva es una recta: 12 horas todo el año. A medida que subes de latitud, la curva oscila más. A 60° el verano supera las 18 horas y el invierno baja de 6. La inclinación del eje es lo que hace que el día se alargue y se acorte.

## 4. Dónde sale el Sol

Para tu pregunta principal necesitas el **azimut de salida**: el ángulo, medido desde el norte hacia el este, del punto del horizonte por donde sale el Sol. 90° es exactamente el este, menos de 90° es al norte del este y más de 90° es al sur.

In [ ]:
def azimut_salida(latitud, dia):
    """Azimut (grados desde el norte hacia el este) de la salida del Sol.
    Horizonte plano y sin refracción. Devuelve nan si el Sol no sale (noche polar) o no se pone."""
    lat = np.radians(latitud)
    dec = np.radians(declinacion(dia))
    c = np.sin(dec) / np.cos(lat)
    return np.degrees(np.arccos(c)) if abs(c) <= 1 else np.nan

fechas = {
    "equinoccio de primavera (~21 mar)": 80,
    "solsticio de junio (~21 jun)": 172,
    "equinoccio de otoño (~23 sep)": 266,
    "solsticio de diciembre (~21 dic)": 355,
}

print(f"Latitud: {mi_latitud}°")
for nombre, dia in fechas.items():
    print(f"{nombre:36s} azimut de salida: {azimut_salida(mi_latitud, dia):6.1f}°   horas de luz: {horas_de_luz(mi_latitud, dia):5.2f}")

### Contrasta el modelo con Stellarium

Abre Stellarium, pon tu latitud y, para cada una de las cuatro fechas, anota el azimut del Sol justo al salir (según la guía de la sesión de astronomía). Escríbelos aquí y ejecuta la celda. Deja `None` en los que aún no hayas medido.

In [ ]:
medidas_stellarium = {
    "equinoccio de primavera (~21 mar)": None,
    "solsticio de junio (~21 jun)": None,
    "equinoccio de otoño (~23 sep)": None,
    "solsticio de diciembre (~21 dic)": None,
}

for nombre, dia in fechas.items():
    modelo = azimut_salida(mi_latitud, dia)
    medido = medidas_stellarium[nombre]
    if medido is None:
        print(f"{nombre:36s} modelo {modelo:6.1f}°   (sin medida todavía)")
    else:
        print(f"{nombre:36s} modelo {modelo:6.1f}°   Stellarium {medido:6.1f}°   diferencia {medido - modelo:+5.1f}°")

**Cómo interpretar la diferencia.** Esperas diferencias de uno o dos grados, no de diez. Hay tres fuentes de discrepancia: la refracción de la atmósfera (el Sol parece estar algo más alto de lo que está), la forma de aproximar la declinación y que los equinoccios y solsticios caen unos días distintos según el año. Si una diferencia es grande, revisa primero que la fecha y la latitud en Stellarium sean las que has puesto aquí.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dias, [azimut_salida(mi_latitud, d) for d in dias])
ax.axhline(90, color="gray", lw=1, ls="--")
ax.text(5, 91.5, "este exacto (90°)", color="gray")
ax.set_xticks(inicio_mes)
ax.set_xticklabels(meses)
ax.set_ylabel("azimut de salida (°)")
ax.set_title(f"Por dónde sale el Sol a lo largo del año (latitud {mi_latitud}°)")
ax.grid(alpha=0.3)
plt.show()

## 5. Para pensar

1. En el ecuador, ¿entre qué azimuts oscila el Sol al salir a lo largo del año? Cambia `mi_latitud` a 0 y compruébalo. ¿Por qué el intervalo es simétrico alrededor de los 90° y es el más estrecho de todos?
2. Pon `mi_latitud = 70` y ejecuta de nuevo las celdas 3 y 4. ¿Qué pasa en junio y en diciembre? ¿Qué significa un `nan`?
3. El eje de la Tierra está inclinado 23,44°. Cambia `OBLICUIDAD` a 0 en la primera celda de la sección 3 y vuelve a dibujar las horas de luz. ¿Qué serían las estaciones en un planeta sin inclinación? Después restáurala.